In [14]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Attention, Concatenate
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')
save_path = '/content/drive/My Drive/Translator_Project/'

# Create directory if it doesn't exist
if not os.path.exists(save_path):
    os.makedirs(save_path)
print(f"Files will be saved to: {save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files will be saved to: /content/drive/My Drive/Translator_Project/


In [15]:
# Sample Data (Replace with your Kaggle dataset)
english_texts = ["nice to meet you", "how are you", "good morning"]
hindi_texts = ["<START> आपसे मिलकर खुशी हुई <END>", "<START> आप कैसे हैं <END>", "<START> शुभ प्रभात <END>"]

# Tokenization for English
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english_texts)
eng_sequences = eng_tokenizer.texts_to_sequences(english_texts)
eng_vocab_size = len(eng_tokenizer.word_index) + 1

# Tokenization for Hindi
hin_tokenizer = Tokenizer(filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n')
hin_tokenizer.fit_on_texts(hindi_texts)
hin_sequences = hin_tokenizer.texts_to_sequences(hindi_texts)
hin_vocab_size = len(hin_tokenizer.word_index) + 1

# Padding
max_eng_len = max(len(x) for x in eng_sequences)
max_hin_len = max(len(x) for x in hin_sequences)

encoder_input_data = pad_sequences(eng_sequences, maxlen=max_eng_len, padding='post')
decoder_input_data = pad_sequences(hin_sequences, maxlen=max_hin_len, padding='post')

# Decoder Output Data (One-step ahead for Teacher Forcing)
decoder_target_data = np.zeros((len(hindi_texts), max_hin_len, hin_vocab_size), dtype='float32')
for i, seq in enumerate(hin_sequences):
    for t, word_id in enumerate(seq):
        if t > 0:
            decoder_target_data[i, t - 1, word_id] = 1.0

print("Data preprocessing complete!")

Data preprocessing complete!


In [16]:
latent_dim = 256

# --- ENCODER ---
encoder_inputs = Input(shape=(None,), name="encoder_input")
enc_emb = Embedding(eng_vocab_size, latent_dim, name="encoder_embedding")(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# --- DECODER ---
decoder_inputs = Input(shape=(None,), name="decoder_input")
dec_emb_layer = Embedding(hin_vocab_size, latent_dim, name="decoder_embedding")
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

# --- ATTENTION MECHANISM ---
attention_layer = Attention(name='attention_layer')
attention_output = attention_layer([decoder_outputs, encoder_outputs])

# Combine decoder output and attention context
concat_layer = Concatenate(axis=-1, name='concat_layer')
decoder_combined_context = concat_layer([decoder_outputs, attention_output])

decoder_dense = Dense(hin_vocab_size, activation='softmax', name="decoder_dense")
decoder_outputs_final = decoder_dense(decoder_combined_context)

# Define the full model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs_final)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 256) │      2,304 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 256) │      3,072 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, None,     │    525,312 │ encoder_embeddin… │
│                     │ 256), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │    525,312 │ decoder_embeddin… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_layer     │ (None, None, 256) │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_layer        │ (None, None, 512) │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention_layer[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, None, 12)  │      6,156 │ concat_layer[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,062,156 (4.05 MB)

 Trainable params: 1,062,156 (4.05 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# Train the model
print("Starting training...")
history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=64,
    epochs=100,
    validation_split=0.2
)

Starting training...
Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.0833 - loss: 1.8652 - val_accuracy: 0.1667 - val_loss: 1.2391
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.2500 - loss: 1.8454 - val_accuracy: 0.1667 - val_loss: 1.2399
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.2500 - loss: 1.8251 - val_accuracy: 0.1667 - val_loss: 1.2404
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.1667 - loss: 1.8032 - val_accuracy: 0.1667 - val_loss: 1.2406
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.2500 - loss: 1.7785 - val_accuracy: 0.1667 - val_loss: 1.2405
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.2500 - loss: 1.7496 - val_accuracy: 0.1667 - val_loss: 1.2399
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.1667 - loss: 1.7150 - val_accuracy: 0.1667 - val_loss: 1.2388
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.1667 - loss: 1.6730 - val_accuracy

In [18]:
print("Saving files to Google Drive...")

# Save Model Weights (Updated for Keras 3)
model.save_weights(save_path + 'model_weights.weights.h5')

# Save Tokenizers
with open(save_path + 'eng_tokenizer.pkl', 'wb') as f:
    pickle.dump(eng_tokenizer, f)

with open(save_path + 'hin_tokenizer.pkl', 'wb') as f:
    pickle.dump(hin_tokenizer, f)

# Save Model Configuration
config = {
    'max_eng_len': max_eng_len,
    'max_hin_len': max_hin_len,
    'eng_vocab_size': eng_vocab_size,
    'hin_vocab_size': hin_vocab_size,
    'latent_dim': latent_dim
}
with open(save_path + 'model_config.pkl', 'wb') as f:
    pickle.dump(config, f)

print("All files saved successfully to:", save_path)

Saving files to Google Drive...
All files saved successfully to: /content/drive/My Drive/Translator_Project/


In [19]:
# --- INFERENCE SETUP ---
# 1. Inference Encoder
encoder_model = Model(encoder_inputs, [encoder_outputs, state_h, state_c])

# 2. Inference Decoder
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_hidden_state_input = Input(shape=(max_eng_len, latent_dim))

decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Reuse the exact same layers trained in Cell 3
dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)

# Reuse attention layers
attention_output2 = attention_layer([decoder_outputs2, decoder_hidden_state_input])
decoder_combined_context2 = concat_layer([decoder_outputs2, attention_output2])
decoder_outputs2 = decoder_dense(decoder_combined_context2)

decoder_model = Model(
    [decoder_inputs, decoder_hidden_state_input] + decoder_states_inputs,
    [decoder_outputs2, state_h2, state_c2]
)

# 3. Translation Function
def decode_sequence(input_text):
    # Preprocess text
    seq = eng_tokenizer.texts_to_sequences([input_text])
    if not seq or not seq[0]:
        return "[Error: Words not in vocabulary]"

    padded_seq = pad_sequences(seq, maxlen=max_eng_len, padding='post')

    # Get encoder output
    enc_outs, h, c = encoder_model.predict(padded_seq, verbose=0)

    # Initialize decoder input with '<START>' token
    start_token = hin_tokenizer.word_index.get('start', hin_tokenizer.word_index.get('<start>', 1))
    end_token = hin_tokenizer.word_index.get('end', hin_tokenizer.word_index.get('<end>', -1))

    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = start_token

    stop_condition = False
    decoded_sentence = ''

    # Generate words one by one
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq, enc_outs, h, c], verbose=0)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = hin_tokenizer.index_word.get(sampled_token_index, '')

        if sampled_token_index == end_token or sampled_word in ['end', '<end>'] or len(decoded_sentence.split()) > max_hin_len:
            stop_condition = True
        else:
            decoded_sentence += ' ' + sampled_word

        # Update target sequence to the predicted word
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

    return decoded_sentence.strip()

# 4. Test it out!
print("--- Testing the Trained Model ---")
for sentence in english_texts:
    translation = decode_sequence(sentence)
    print(f"English: {sentence}")
    print(f"Hindi:   {translation}\n")

--- Testing the Trained Model ---
English: nice to meet you
Hindi:   आपसे मिलकर खुशी हुई

English: how are you
Hindi:   आप कैसे हैं

English: good morning
Hindi:   आप कैसे

